<div class="alert alert-info" style="background-color:#001b79; color:white; padding:0px 10px; border-radius:5px;"><h2 style='margin:10px 5px'>Recommendation Engine UBCF CBCF </h2>
</div>

> Is a tool, that allows algorithm developers predict what user may or may not like  among a list of given items.

> Help users discover products or content that we may not come across otherwise

### RE works ideally 2 ways:		
####	1. User Base filtering (UB)/User Based Collaborative Filtering (UBCF):	
		Building a model based on users past behaviour as well as similar decision made by other users.
		This model is used to predict items that the users may have an interset in.
		But in this tend to recommend only popular items and fail to recommend new or unique taste one
		
		
		
####	2. Content- Based Filtering(CB)	
		Utilizes series of discrete characterestics of an item in order to recommend additional items to users with similar properties.
		Here items characterstics are identified , similar characterstics other items will be recommended.
        Here unique taste one will be recommended but never recommend only popular items outside users content profile
		
		
####	3. Hybrid Recommendation system:	
		This is combined feature of above both filtering, this gives better result




<div class="alert alert-info" style="background-color:#001b32; color:white; padding:0px 10px; border-radius:5px;"><h2 style='margin:10px 5px'>Problem  Statement </h2>
</div>

### Create Cosine similarity matrix for users and movies and predict a user-movie recommendation model(based on the difference from mean rating)

##  1. Import all Librariy


In [28]:
import pandas as pd
import numpy as np

In [29]:
df = pd.read_csv(r'..\Dataset\Recommend.csv',names=['user_id', 'movie_id', 'rating', 'timestamp'])
df.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


## 2. Data Exploration

In [30]:
# finding unique number of users and movies
#n_users = len(df['user_id'].unique())

n_users = df.user_id.unique().shape[0] 
n_movies = df.movie_id.unique().shape[0]
print(n_users)
print(n_movies)

943
1682


## 3. Building Machine Learning Recommendation engine - Collaborative Filtering

### 3.1 Train Test Split


- Recommendation Systems are difficult to evaluate, but we will still show you how to evaluate them. In order to do this, we'll split our data into two sets. However, we won't do our classic X_train,X_test,y_train,y_test split. Instead we can actually just segement the data into two sets of data:

In [31]:
# split the data into training and testing
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(df, test_size=0.25)

In [32]:
#Priovides matrix with no rows= no of unique users,  no col= no of unique movies with all 0
#lets create an array of ratings for training data
# train array(user_id, movie_id) containing ratings such that..
# [user_id_index, movie_id_index] -> rating
# Here assuming user id and movie id are sequence no from 1

train_data_matrix = np.zeros((n_users, n_movies))
print(train_data_matrix)
print("########")

#In this for loop, we will assign rating values to users vs movie position from 0 to actual rating
# itertuples: returns a named tuple with indexvalue and colvalues
#  line is : its tuple of index value and all col value => user_id_order	isbn_id	rating	book_title	book_author	year_of_publication	publisher	isbn	user_id
 # print(line)   # Pandas(Index=10223, user_id=382, movie_id=546, rating=2, timestamp=875946234)
    
# Here line[3]  is 4th value in tuple which is rating
# line[1]-1 provide user id value , line[2]-1 book_id value. user_id and book_id are continous value in line datset
# train_data_matrix[0,0] --> 1st row 1st col  position in matrix
# train_data_matrix[0,1] --> 1st row 2nd col  position in matrix

for line in train_data.itertuples():
    #[user_id index, movie_id index] = given rating.
    # line is : its tuple of index value and all col value => user_id movie_id rating timestamp
    # print(line)   # Pandas(Index=10223, user_id=382, movie_id=546, rating=2, timestamp=875946234)
     # This assign rating value in each matrix position
    train_data_matrix[line[1]-1, line[2]-1] = line[3] 
train_data_matrix

#Below 1 st row 1st value is 5. means, user_id=1 provided  rating =5  for movie_id=1 
#Below 1st row 2nd value is 3. means, user_id=1 provided  rating =3  for movie_id=2
#Below 2nd row 1st value is 4. means, user_id=2 provided  rating =4  for movie_id=1

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
########


array([[5., 3., 4., ..., 0., 0., 0.],
       [4., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [5., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 5., 0., ..., 0., 0., 0.]])

In [33]:
# do the same for testing data

test_data_matrix = np.zeros((n_users, n_movies))
for line in test_data.itertuples():
    #[user_id index, movie_id index] = given rating.
    # line is : user_id movie_id rating timestamp
    test_data_matrix[line[1]-1, line[2]-1] = line[3]
test_data_matrix

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

### 3.2.2 Apply pairwise_distances function with cosine similarity
You can use the pairwise_distances function from sklearn to calculate the cosine similarity. Note, the output will range from 0 to 1 since the ratings are all positive

In [34]:
from sklearn.metrics import pairwise_distances

# create cosine similarities for users and movies and then predict
#.. a user-movie recommendation model(based on the difference from mean rating)

user_similarity = pairwise_distances(train_data_matrix, metric='cosine')
movie_similarity = pairwise_distances(train_data_matrix.T, metric='cosine')
#This provides mean rating in user wise for considering all movie rating , total 943 users, so 943 rows will be given as mean_user_rating

#mean(all column rating, axis=1 -->mean calculated based on col wise, considering all col value/all movie )
mean_user_rating = train_data_matrix.mean(axis=1)[:, np.newaxis] 

#print(mean_user_rating)
print(train_data_matrix.shape)
print(mean_user_rating.shape)
print(mean_user_rating[:5])

# difference of actual rating - mean rating
ratings_diff = (train_data_matrix - mean_user_rating) 


(943, 1682)
(943, 1)
[[0.41260404]
 [0.10998811]
 [0.06956005]
 [0.04994055]
 [0.23781213]]


### 3.2.3 Make Prediction

    - Next step is to make predictions

In [35]:
# User User-Movie based filtering  Prediction
# User_prediction
#pred dormula: mean_user_rating +  (user_similarity  * ratings_diff ) / (This provides sum of 'user_similarity' in user wise for considering all movie rating )
user_pred = mean_user_rating + user_similarity.dot(ratings_diff) / np.array([np.abs(user_similarity).sum(axis=1)]).T
print(user_pred)

# Content movie-movie based filtering  Prediction
# item_prediction
movie_pred = train_data_matrix.dot(movie_similarity) / np.array([np.abs(movie_similarity).sum(axis=1)])
print(movie_pred)

[[ 1.62109871  0.52923501  0.43276027 ...  0.26125351  0.26361444
   0.26125351]
 [ 1.40777064  0.27775776  0.13657984 ... -0.05983071 -0.05647692
  -0.05983071]
 [ 1.42374935  0.23223004  0.10032579 ... -0.10055617 -0.09731241
  -0.10055617]
 ...
 [ 1.26930787  0.19195832  0.06095361 ... -0.1267746  -0.12373946
  -0.1267746 ]
 [ 1.43066953  0.28489116  0.17491963 ... -0.0222117  -0.01937571
  -0.0222117 ]
 [ 1.48367578  0.37219126  0.28177723 ...  0.107646    0.11004365
   0.107646  ]]
[[0.33620111 0.35405049 0.3678864  ... 0.41284949 0.40511533 0.41284949]
 [0.09330093 0.10783616 0.10425372 ... 0.11005354 0.11024963 0.11005354]
 [0.06781042 0.06980975 0.06831434 ... 0.06960143 0.06956283 0.06960143]
 ...
 [0.02737995 0.03516149 0.03462284 ... 0.03985723 0.03904565 0.03985723]
 [0.11701692 0.12586476 0.13480322 ... 0.13979774 0.13766967 0.13979774]
 [0.20566664 0.20759225 0.22858155 ... 0.26055919 0.25357167 0.26055919]]


## 4. Model Evaluation 
 - There are many evaluation metrics but one of the most popular metric used to evaluate accuracy of predicted ratings is *Root Mean Squared Error (RMSE)*.

In [36]:
from sklearn.metrics import mean_squared_error
from math import sqrt
def rmse(pred, test):
    pred = pred[test.nonzero()].flatten() 
    test = test[test.nonzero()].flatten()
    return sqrt(mean_squared_error(pred, test))

In [37]:

print('User-based CF RMSE: ' + str(rmse(user_pred, test_data_matrix)))
print('Item-based CF RMSE: ' + str(rmse(movie_pred, test_data_matrix)))

User-based CF RMSE: 3.1156093177380355
Item-based CF RMSE: 3.4410834703840605


## 5. Conclusion

> Both the approach yield almost same result

> How to understand predicted value is not clear yet. yet to know more